# Train TRRUST Classifiers

Pipeline:

1. **Load** gene embeddings + TRRUST data.
2. **(Optional)** Restrict training data to a gene subset from a file.
3. **(Optional)** Write the current gene universe to a file for cross-model comparisons.
4. **Hyperparameter sweep with 5-fold CV per config** — 5 LRs × 4 epoch counts
   for binary + ternary. For each config we run stratified 5-fold CV and save
   per-fold train/test loss curves, per-fold classification reports, a summary
   CSV with mean/std accuracy and macro F1, and a heatmap of **mean macro F1**
   across LR × epochs under `REPORTS_DIR`.
5. **Final results** — re-run 5-fold CV at the chosen LR / epoch count for
   binary + ternary (inspect the sweep heatmap, then set `*_LR` / `*_EPOCHS`).
6. **Save final results** as a pickle for downstream analysis.

This notebook is model-agnostic — point `EMBEDDINGS_PATH` at any h5ad file produced
by `src/scgpt/encode.py` or `src/geneformer/encode.py`.


In [2]:
import itertools
import pickle
import warnings

warnings.filterwarnings("ignore")

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import classification_report

from src.trrust import (
    BINARY_LABEL_NAMES,
    BINARY_LABELS,
    TERNARY_LABEL_NAMES,
    TERNARY_LABELS,
    cross_validate,
    filter_data_by_genes,
    load_binary_trrust_data,
    load_gene_embeddings,
    load_ternary_trrust_data,
)


## Configuration

Change `EMBEDDINGS_PATH` to swap foundation models, and point `REPORTS_DIR` /
`CV_RESULTS_PATH` at model-specific output locations. Set `GENE_SUBSET_PATH` to
restrict training to pairs where both TF and target appear in a gene-list file
(see `data/scgpt_binary_genes.txt`).

`LR_GRID` and `EPOCH_GRID` define the hyperparameter sweep; each `(lr, epochs)`
config is evaluated with 5-fold stratified CV. After inspecting the mean macro
F1 heatmap, fill in `BINARY_LR` / `BINARY_EPOCHS` / `TERNARY_LR` /
`TERNARY_EPOCHS` before running the final-results cells.


In [3]:
EMBEDDINGS_PATH = Path("../data/embeddings/geneformer_cd20.h5ad")
TRRUST_PATH = Path("../data/trrust_rawdata.human.tsv")
REPORTS_DIR = Path("../reports/geneformer/hp_tuning_cd20")
CV_RESULTS_PATH = Path("../data/geneformer_cv_results_cd20.pkl")
GENE_SUBSET_PATH: Path | None = Path("../data/scgpt_binary_genes.txt")  # e.g. Path("../data/scgpt_binary_genes.txt") or None

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
BATCH_SIZE = 64

LR_GRID = [1e-2, 1e-3, 1e-4, 1e-5, 1e-6]
EPOCH_GRID = [8, 12, 20, 50]


print(f"Device: {DEVICE}")
print(f"Embeddings: {EMBEDDINGS_PATH}")
print(f"TRRUST: {TRRUST_PATH}")
print(f"Reports dir: {REPORTS_DIR}")
print(f"CV results: {CV_RESULTS_PATH}")
print(f"Gene subset: {GENE_SUBSET_PATH}")


Device: cuda
Embeddings: ../data/embeddings/geneformer_cd20.h5ad
TRRUST: ../data/trrust_rawdata.human.tsv
Reports dir: ../reports/geneformer/hp_tuning_cd20
CV results: ../data/geneformer_cv_results_cd20.pkl
Gene subset: ../data/scgpt_binary_genes.txt


## Load embeddings and TRRUST data

In [4]:
gene_embeddings = load_gene_embeddings(EMBEDDINGS_PATH)
embsize = next(iter(gene_embeddings.values())).shape[0]

binary_data = load_binary_trrust_data(TRRUST_PATH, gene_embeddings)
ternary_data = load_ternary_trrust_data(TRRUST_PATH, gene_embeddings)

print(f"Gene embeddings: {len(gene_embeddings)} genes, {embsize}d")
print(f"Binary samples: {len(binary_data.records)}")
print(f"Ternary samples: {len(ternary_data.records)}")


Gene embeddings: 11539 genes, 1152d
Binary samples: 9048
Ternary samples: 3537


## (Optional) Restrict training data to a gene subset

If `GENE_SUBSET_PATH` is set, keep only TRRUST records where both TF and target are
listed in the file (one gene symbol per line). A no-op when `GENE_SUBSET_PATH is None`.


In [5]:
if GENE_SUBSET_PATH is not None:
    subset_genes = {
        line.strip()
        for line in Path(GENE_SUBSET_PATH).read_text().splitlines()
        if line.strip()
    }
    print(f"Subset file: {GENE_SUBSET_PATH} ({len(subset_genes)} genes)")

    before_binary = len(binary_data.records)
    before_ternary = len(ternary_data.records)
    binary_data = filter_data_by_genes(binary_data, subset_genes)
    ternary_data = filter_data_by_genes(ternary_data, subset_genes)
    print(f"Binary records: {before_binary} -> {len(binary_data.records)}")
    print(f"Ternary records: {before_ternary} -> {len(ternary_data.records)}")
else:
    print("GENE_SUBSET_PATH is None — using all records.")


Subset file: ../data/scgpt_binary_genes.txt (1803 genes)
Binary records: 9048 -> 7919
Ternary records: 3537 -> 3106


## (Optional) Save current gene lists for future cross-model runs

Write the union of TFs + targets from the current `binary_data` / `ternary_data` to
two `.txt` files so another run (e.g. a different embedding model) can be restricted
to the same gene universe via `GENE_SUBSET_PATH` above. Edit the output paths for
the model you are running.


In [12]:
BINARY_GENE_FILE = Path("../data/scgpt_binary_genes.txt")
TERNARY_GENE_FILE = Path("../data/scgpt_ternary_genes.txt")

binary_genes = sorted({g for r in binary_data.records for g in (r.tf, r.target)})
BINARY_GENE_FILE.write_text("\n".join(binary_genes) + "\n")
print(f"Saved {len(binary_genes)} binary genes to {BINARY_GENE_FILE}")

ternary_genes = sorted({g for r in ternary_data.records for g in (r.tf, r.target)})
TERNARY_GENE_FILE.write_text("\n".join(ternary_genes) + "\n")
print(f"Saved {len(ternary_genes)} ternary genes to {TERNARY_GENE_FILE}")


Saved 1803 binary genes to ../data/scgpt_binary_genes.txt
Saved 1561 ternary genes to ../data/scgpt_ternary_genes.txt


## Hyperparameter sweep helpers

Each config runs 5-fold stratified cross-validation (seed=42) so rankings are
robust to any single split. For every `(lr, epochs)` combination we save a
single plot with overlaid per-fold train/test loss curves, a text report with
per-fold classification reports plus mean ± std accuracy / macro F1, and a row
in `summary.csv`. After the loop, a heatmap of **mean macro F1** over LR ×
epochs is saved alongside the per-config outputs.


In [6]:
def _save_loss_plot(cv_result, lr, epochs, title_prefix, out_path):
    plt.figure(figsize=(8, 4))
    xs = range(1, epochs + 1)
    colors = plt.get_cmap("tab10").colors
    for i, fold in enumerate(cv_result.per_fold):
        color = colors[i % len(colors)]
        plt.plot(xs, fold.train_losses, color=color, linestyle="-",
                 label=f"Fold {i + 1} train")
        plt.plot(xs, fold.test_losses, color=color, linestyle="--",
                 label=f"Fold {i + 1} test")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{title_prefix} — lr={lr:.0e}, epochs={epochs}")
    plt.legend(fontsize=8, ncol=2)
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def _fold_macro_f1s(cv_result):
    return np.array([
        fold.classification_report["macro avg"]["f1-score"]
        for fold in cv_result.per_fold
    ])


def _save_text_report(cv_result, label_names, out_path):
    target_names = [label_names[i] for i in range(len(label_names))]
    name_to_label = {name: idx for idx, name in label_names.items()}
    lines = []
    for i, fold in enumerate(cv_result.per_fold):
        preds = fold.gene_predictions
        true_ids = preds["true_relationship"].map(name_to_label).to_numpy()
        pred_ids = preds["predicted_relationship"].map(name_to_label).to_numpy()
        lines.append(f"=== Fold {i + 1} ===")
        lines.append(classification_report(
            true_ids, pred_ids, target_names=target_names, zero_division=0
        ))
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    lines.append(f"Mean accuracy: {accs.mean():.3f} ± {accs.std():.3f}")
    lines.append(f"Mean macro F1: {f1s.mean():.3f} ± {f1s.std():.3f}")
    out_path.write_text("\n".join(lines))


def _summary_row(lr, epochs, cv_result, label_names):
    accs = np.array(cv_result.fold_accuracies)
    f1s = _fold_macro_f1s(cv_result)
    row = {
        "lr": lr,
        "epochs": epochs,
        "mean_accuracy": accs.mean(),
        "std_accuracy": accs.std(),
        "mean_macro_f1": f1s.mean(),
        "std_macro_f1": f1s.std(),
    }
    agg = cv_result.aggregate_classification_report
    row["pooled_accuracy"] = agg["accuracy"]
    for name in ("macro avg", "weighted avg"):
        for metric in ("precision", "recall", "f1-score"):
            row[f"pooled_{name}_{metric}"] = agg[name][metric]
    for i in range(len(label_names)):
        name = label_names[i]
        row[f"pooled_{name}_precision"] = agg[name]["precision"]
        row[f"pooled_{name}_recall"] = agg[name]["recall"]
        row[f"pooled_{name}_f1"] = agg[name]["f1-score"]
    return row


def _save_f1_heatmap(summary_df, title, out_path):
    pivot = summary_df.pivot(
        index="lr", columns="epochs", values="mean_macro_f1"
    ).sort_index(ascending=False)
    fig, ax = plt.subplots(figsize=(6, 4))
    im = ax.imshow(pivot.values, aspect="auto", cmap="viridis")
    ax.set_xticks(range(len(pivot.columns)))
    ax.set_xticklabels(pivot.columns)
    ax.set_yticks(range(len(pivot.index)))
    ax.set_yticklabels([f"{v:.0e}" for v in pivot.index])
    ax.set_xlabel("Epochs")
    ax.set_ylabel("Learning rate")
    ax.set_title(title)
    for i in range(pivot.shape[0]):
        for j in range(pivot.shape[1]):
            ax.text(j, i, f"{pivot.values[i, j]:.2f}", ha="center", va="center",
                    color="white" if pivot.values[i, j] < pivot.values.mean() else "black")
    fig.colorbar(im, ax=ax, label="mean macro F1")
    plt.tight_layout()
    plt.savefig(out_path, dpi=120)
    plt.close()


def run_hp_sweep(
    data,
    *,
    label_map,
    label_names,
    use_class_weights,
    out_dir,
    title_prefix,
    n_splits=5,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    rows = []
    for lr, epochs in itertools.product(LR_GRID, EPOCH_GRID):
        cv_result = cross_validate(
            data,
            embsize=embsize,
            label_map=label_map,
            lr=lr,
            epochs=epochs,
            batch_size=BATCH_SIZE,
            use_class_weights=use_class_weights,
            n_splits=n_splits,
            device=DEVICE,
            seed=42,
        )
        tag = f"lr{lr:.0e}_e{epochs}"
        _save_loss_plot(cv_result, lr, epochs, title_prefix, out_dir / f"loss_{tag}.png")
        _save_text_report(cv_result, label_names, out_dir / f"report_{tag}.txt")
        row = _summary_row(lr, epochs, cv_result, label_names)
        rows.append(row)
        print(f"  {tag}: mean acc={row['mean_accuracy']:.3f} ± {row['std_accuracy']:.3f}, "
              f"mean macro F1={row['mean_macro_f1']:.3f} ± {row['std_macro_f1']:.3f}")

    summary = pd.DataFrame(rows)
    summary.to_csv(out_dir / "summary.csv", index=False)
    _save_f1_heatmap(
        summary,
        f"{title_prefix} mean macro F1 (5-fold CV)",
        out_dir / "f1_heatmap.png",
    )
    return summary


## Hyperparameter sweep — binary classifier

20 configs (5 LRs × 4 epoch counts), each run through 5-fold stratified CV.
No class weights (binary data is balanced by construction). Outputs land in
`REPORTS_DIR/binary/`.


In [7]:
binary_summary = run_hp_sweep(
    binary_data,
    label_map=BINARY_LABELS,
    label_names=BINARY_LABEL_NAMES,
    use_class_weights=False,
    out_dir=REPORTS_DIR / "binary",
    title_prefix="Binary",
)
binary_summary.sort_values("mean_macro_f1", ascending=False).head()


  lr1e-02_e8: mean acc=0.735 ± 0.010, mean macro F1=0.734 ± 0.011
  lr1e-02_e12: mean acc=0.732 ± 0.010, mean macro F1=0.731 ± 0.009
  lr1e-02_e20: mean acc=0.724 ± 0.007, mean macro F1=0.723 ± 0.007
  lr1e-02_e50: mean acc=0.726 ± 0.004, mean macro F1=0.726 ± 0.004
  lr1e-03_e8: mean acc=0.742 ± 0.006, mean macro F1=0.742 ± 0.006
  lr1e-03_e12: mean acc=0.737 ± 0.016, mean macro F1=0.737 ± 0.016
  lr1e-03_e20: mean acc=0.730 ± 0.015, mean macro F1=0.729 ± 0.015
  lr1e-03_e50: mean acc=0.733 ± 0.013, mean macro F1=0.733 ± 0.013
  lr1e-04_e8: mean acc=0.728 ± 0.006, mean macro F1=0.728 ± 0.006
  lr1e-04_e12: mean acc=0.740 ± 0.005, mean macro F1=0.740 ± 0.005
  lr1e-04_e20: mean acc=0.743 ± 0.004, mean macro F1=0.743 ± 0.004
  lr1e-04_e50: mean acc=0.742 ± 0.009, mean macro F1=0.742 ± 0.009
  lr1e-05_e8: mean acc=0.646 ± 0.011, mean macro F1=0.646 ± 0.011
  lr1e-05_e12: mean acc=0.666 ± 0.009, mean macro F1=0.666 ± 0.009
  lr1e-05_e20: mean acc=0.693 ± 0.007, mean macro F1=0.692 ± 0.007

,lr,epochs,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,pooled_accuracy,pooled_macro avg_precision,pooled_macro avg_recall,pooled_macro avg_f1-score,pooled_weighted avg_precision,pooled_weighted avg_recall,pooled_weighted avg_f1-score,pooled_None_precision,pooled_None_recall,pooled_None_f1,pooled_Relationship_precision,pooled_Relationship_recall,pooled_Relationship_f1
10,0.0001,20,0.743402,0.003752,0.743230,0.003743,0.743402,0.743315,0.743221,0.743256,0.743377,0.743402,0.743377,0.740192,0.734089,0.737128,0.746437,0.752353,0.749383
11,0.0001,50,0.741885,0.008697,0.741687,0.008744,0.741887,0.741820,0.741660,0.741710,0.741861,0.741887,0.741844,0.739755,0.730224,0.734959,0.743885,0.753096,0.748462
4,0.0010,8,0.741888,0.005783,0.741593,0.005661,0.741887,0.741887,0.741575,0.741647,0.741887,0.741887,0.741803,0.741902,0.725844,0.733785,0.741873,0.757306,0.749510
9,0.0001,12,0.740118,0.005153,0.739913,0.005252,0.740119,0.740027,0.739941,0.739974,0.740093,0.740119,0.740095,0.736692,0.730997,0.733833,0.743363,0.748886,0.746114
5,0.0010,12,0.737089,0.016059,0.736506,0.016461,0.737088,0.737442,0.736539,0.736617,0.737334,0.737088,0.736838,0.742911,0.708838,0.725475,0.731973,0.764240,0.747759


## Hyperparameter sweep — ternary classifier

20 configs, each run through 5-fold stratified CV with inverse-frequency class
weights (ternary data is imbalanced). Outputs land in `REPORTS_DIR/ternary/`.


In [8]:
ternary_summary = run_hp_sweep(
    ternary_data,
    label_map=TERNARY_LABELS,
    label_names=TERNARY_LABEL_NAMES,
    use_class_weights=True,
    out_dir=REPORTS_DIR / "ternary",
    title_prefix="Ternary",
)
ternary_summary.sort_values("mean_macro_f1", ascending=False).head()


  lr1e-02_e8: mean acc=0.515 ± 0.012, mean macro F1=0.507 ± 0.015
  lr1e-02_e12: mean acc=0.516 ± 0.014, mean macro F1=0.509 ± 0.017
  lr1e-02_e20: mean acc=0.515 ± 0.012, mean macro F1=0.507 ± 0.015
  lr1e-02_e50: mean acc=0.517 ± 0.013, mean macro F1=0.509 ± 0.016
  lr1e-03_e8: mean acc=0.533 ± 0.011, mean macro F1=0.522 ± 0.015
  lr1e-03_e12: mean acc=0.537 ± 0.007, mean macro F1=0.527 ± 0.011
  lr1e-03_e20: mean acc=0.538 ± 0.004, mean macro F1=0.527 ± 0.007
  lr1e-03_e50: mean acc=0.539 ± 0.005, mean macro F1=0.527 ± 0.007
  lr1e-04_e8: mean acc=0.507 ± 0.011, mean macro F1=0.495 ± 0.011
  lr1e-04_e12: mean acc=0.516 ± 0.010, mean macro F1=0.502 ± 0.010
  lr1e-04_e20: mean acc=0.529 ± 0.011, mean macro F1=0.514 ± 0.011
  lr1e-04_e50: mean acc=0.536 ± 0.015, mean macro F1=0.519 ± 0.014
  lr1e-05_e8: mean acc=0.416 ± 0.012, mean macro F1=0.408 ± 0.012
  lr1e-05_e12: mean acc=0.428 ± 0.016, mean macro F1=0.419 ± 0.015
  lr1e-05_e20: mean acc=0.456 ± 0.018, mean macro F1=0.446 ± 0.018

,lr,epochs,mean_accuracy,std_accuracy,mean_macro_f1,std_macro_f1,pooled_accuracy,pooled_macro avg_precision,pooled_macro avg_recall,pooled_macro avg_f1-score,...,pooled_weighted avg_f1-score,pooled_Activation_precision,pooled_Activation_recall,pooled_Activation_f1,pooled_Repression_precision,pooled_Repression_recall,pooled_Repression_f1,pooled_None_precision,pooled_None_recall,pooled_None_f1
7,0.0010,50,0.539276,0.005373,0.527334,0.007359,0.539279,0.529146,0.526657,0.527699,...,0.539439,0.572193,0.594444,0.583106,0.403846,0.404819,0.404332,0.611399,0.580709,0.595659
5,0.0010,12,0.537349,0.007311,0.526629,0.010937,0.537347,0.528120,0.526451,0.527176,...,0.537801,0.570762,0.582540,0.576591,0.411137,0.418072,0.414576,0.602459,0.578740,0.590361
6,0.0010,20,0.538314,0.003944,0.526504,0.006502,0.538313,0.528594,0.526084,0.527132,...,0.538610,0.571210,0.592063,0.581450,0.405018,0.408434,0.406719,0.609553,0.577756,0.593229
4,0.0010,8,0.533166,0.011146,0.522328,0.015162,0.533162,0.524016,0.521892,0.522777,...,0.533801,0.568992,0.582540,0.575686,0.403055,0.413253,0.408090,0.600000,0.569882,0.584553
11,0.0001,50,0.535739,0.014671,0.518886,0.013948,0.535737,0.520498,0.519472,0.519039,...,0.532086,0.561391,0.602381,0.581164,0.412831,0.356627,0.382676,0.587271,0.599409,0.593278


## Final results — binary classifier

Re-run 5-fold CV at the chosen `(BINARY_LR, BINARY_EPOCHS)` (read off the sweep
heatmap) to produce the final reportable per-fold accuracies and the pooled
(aggregate) classification report.


In [11]:
BINARY_LR = 1e-4
BINARY_EPOCHS = 20
BINARY_FOLDS = 5

In [12]:
binary_cv = cross_validate(
    binary_data,
    embsize=embsize,
    label_map=BINARY_LABELS,
    lr=BINARY_LR,
    epochs=BINARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=False,
    n_splits=BINARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(binary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(binary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.742
Fold 2: accuracy=0.743
Fold 3: accuracy=0.751
Fold 4: accuracy=0.740
Fold 5: accuracy=0.742

              precision  recall  f1-score   support
None              0.740   0.734     0.737  3881.000
Relationship      0.746   0.752     0.749  4038.000
accuracy          0.743   0.743     0.743     0.743
macro avg         0.743   0.743     0.743  7919.000
weighted avg      0.743   0.743     0.743  7919.000


## Final results — ternary classifier

Re-run 5-fold CV at the chosen `(TERNARY_LR, TERNARY_EPOCHS)` with inverse-
frequency class weights to produce the final reportable per-fold accuracies and
pooled classification report.


In [17]:
TERNARY_LR = 1e-3
TERNARY_EPOCHS = 50
TERNARY_FOLDS = 5

In [18]:
ternary_cv = cross_validate(
    ternary_data,
    embsize=embsize,
    label_map=TERNARY_LABELS,
    lr=TERNARY_LR,
    epochs=TERNARY_EPOCHS,
    batch_size=BATCH_SIZE,
    use_class_weights=True,
    n_splits=TERNARY_FOLDS,
    device=DEVICE,
)

for i, acc in enumerate(ternary_cv.fold_accuracies):
    print(f"Fold {i + 1}: accuracy={acc:.3f}")
print()
print(pd.DataFrame(ternary_cv.aggregate_classification_report).T.round(3))


Fold 1: accuracy=0.548
Fold 2: accuracy=0.535
Fold 3: accuracy=0.543
Fold 4: accuracy=0.535
Fold 5: accuracy=0.536

              precision  recall  f1-score   support
Activation        0.572   0.594     0.583  1260.000
Repression        0.404   0.405     0.404   830.000
None              0.611   0.581     0.596  1016.000
accuracy          0.539   0.539     0.539     0.539
macro avg         0.529   0.527     0.528  3106.000
weighted avg      0.540   0.539     0.539  3106.000


## Save final results

Pickle a `{"binary": CrossValidationResult, "ternary": CrossValidationResult}`
dict to `CV_RESULTS_PATH` for downstream analysis notebooks to load.


In [11]:
CV_RESULTS_PATH.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "binary": binary_cv,
    "ternary": ternary_cv,
    "embeddings_path": str(EMBEDDINGS_PATH),
    "gene_subset_path": str(GENE_SUBSET_PATH) if GENE_SUBSET_PATH else None,
}
with open(CV_RESULTS_PATH, "wb") as f:
    pickle.dump(payload, f)

print(f"Saved CV results to {CV_RESULTS_PATH}")
print(f"  binary:  lr={binary_cv.config['lr']}, epochs={binary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(binary_cv.fold_accuracies):.3f}")
print(f"  ternary: lr={ternary_cv.config['lr']}, epochs={ternary_cv.config['epochs']}, "
      f"mean fold acc={np.mean(ternary_cv.fold_accuracies):.3f}")


Saved CV results to ../data/scgpt_cv_results_cd20.pkl
  binary:  lr=0.0001, epochs=50, mean fold acc=0.669
  ternary: lr=0.0001, epochs=50, mean fold acc=0.398
